In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats


# Multi-Period Binomial Tree

### Parameters

In [ ]:
""" 
Set bimomial model parameters below. 
Here we choose the parameters associated to the Monte-Carlo Exercise 4.9.
You are encouraged to modify these and play around with the code.
"""

T = 90 # number of periods
S0 = 50 # initial stock price
r = 0.01 # risk-free rate

## specify u and d values below 

# for q close to 0.5
# u = 0.1
# d = -0.1 

# ## for q close to 1 (below parameters as in Ex 4.9)
u = 0.0105 # up move volatility
d = -0.0104 # down move volatility

### Simulate and plot trajectories under risk-neutral probability

In [ ]:
q = (r-d)/(u-d) # Q(G)
# N = 10**4 # number of sample paths to simulate
N = 100
print('q=',q)
print('N=',f"{N:.0e}")
print('Total possible paths = ', f"{2**T:.2e}")


Below we simple N sample paths of the stock price

To do this we generate a N x T matrix called 'omegas' of 0s and 1s coming from a 
Bernoulli (i.e. Binomial with n=1) distribution with probability of success q.

omegas(n,t) = 1 then means that the outcome of the nth experiment at time t is G
and if it is zero then it is B.

Using this we then compute the number of up moves k_t(omega) matrix.

Finally we compute the N x T matrix of correspond stock trajectories S (each row is a sample path)

The trajectories are then plotted

In [ ]:
omegas = np.random.binomial(1,q,(N,T)) # random draws
k = np.cumsum(omegas,axis=1) # entry (n,t) is k_t(omega^n)
time = np.linspace(np.ones(N),T*np.ones(N),T).T # N x T matrix of time point
S = S0*(1+u)**k*(1+d)**(time-k) # entry (n,t) is S_t(omega^n) = S_0(1+u)^k_t(omega^n)*(1+d)^(t-k_t(omega^n))
S = np.append(S0*np.ones(N)[:,None],S,axis=1) # adds initial value to the first column (time 0)

In [ ]:
fig = plt.subplots(figsize=(8,6))
for i in range(N):
    plt.plot(S[i,:],alpha=0.5)
plt.grid(True)
plt.yscale('log') # optional, plots y scale on log axis
plt.plot()

## Lookback Call Option

We price a lookback call option, which has payoff $V_T = S_T - \min_{t \in \mathbb{T}}S_t$

### Time zero pricing

In [ ]:
running_min = np.minimum.accumulate(S,axis=1) # computes running min in matrix format
lookback_payoff = S[:,-1] - running_min[:,-1] # computes payoffs for each sample
lookback_V0 = np.mean(lookback_payoff/(1+r)**T) # obtains estimate
z = 0.69847 # z-value for 95% confidence interval

print(r'Our estimated price is',lookback_V0)
print(r'Our 95% confidence interval is [',lookback_V0 - z*np.std(lookback_payoff/(1+r)**T)/np.sqrt(N),',',lookback_V0 + z*np.std(lookback_payoff/(1+r)**T)/np.sqrt(N),']') 
print('If we want to shrink our confidence interval by a factor of 10, we would need on average 100 = 10^2 more samples')

### Time one pricing

We now modify the above to also get a time 1 price estimate at outcomes G and B respectively.

To this end we refer to the first columns of the omegas matrix, which tells us which paths went through the repsective nodes.

Then we separately average the payoffs, discounted to time t=1 conditional on going through the up and down node at time t=1.

Once we have $V_1(G)$ and $V_1(B)$ we can estimate $\theta_0 = \frac{V_1(G)-V_1(B)}{S_1(G)-S_1(B)}$.

In [ ]:
up_indicator = k[:,0] # time one indicator variable for G or B
num_up = np.sum(k[:,0]) # total number of paths with omega_1 = G
 
lookback_V1_G = np.sum(lookback_payoff[up_indicator==1]/(1+r)**(T-1))/num_up
lookback_V1_B = np.sum(lookback_payoff[up_indicator==0]/(1+r)**(T-1))/(N-num_up)

lookback_theta0 = (lookback_V1_G-lookback_V1_B)/(S0*(u-d))

print('Our estimated time one prices are V_1(G)=', lookback_V1_G, ' and V_1(B) =', lookback_V1_B)
print('The replicating portfolio at time zero is theta_0 =', lookback_theta0)

## Asian Option

Here we repeat the above for the asian option $V_T = (\frac{1}{T}\sum_{t=1}^T S_t-K)_+$ with strike price $K=50$.

### Time zero pricing

In [ ]:
K = 50
average_price = np.mean(S,axis=1)
asian_payoff = np.where(average_price-K < 0, 0, average_price-K)
asian_V0 = np.mean(asian_payoff/(1+r)**T)

z = 0.69847 # z-value for 95% confidence interval

print(r'Our estimated price is',asian_V0)
print(r'Our 95% confidence interval is [',asian_V0 - z*np.std(asian_payoff/(1+r)**T)/np.sqrt(N),',',asian_V0 + z*np.std(asian_payoff/(1+r)**T)/np.sqrt(N),']') 
print('If we want to shrink our confidence interval by a factor of 10, we would need on average 100 = 10^2 more samples')

### Time one pricing

In [ ]:
up_indicator = k[:,0] # time one indicator variable for G or B
num_up = np.sum(k[:,0]) # total number of paths with omega_1 = G

asian_V1_G = np.sum(asian_payoff[up_indicator==1]/(1+r)**(T-1))/num_up
asian_V1_B = np.sum(asian_payoff[up_indicator==0]/(1+r)**(T-1))/(N-num_up)

asian_theta0 = (asian_V1_G-asian_V1_B)/(S0*(u-d))

print('Our estimated time one prices are V_1(G)=', asian_V1_G, ' and V_1(B) =', asian_V1_B)
print('The replicating portfolio at time zero is theta_0 =', asian_theta0)